### **Problem Statement**

Time series analytics and model using PySpark and Prophet

References :

**https://www.statsmodels.org/dev/generated/statsmodels.tsa.seasonal.seasonal_decompose.html**

**https://facebook.github.io/prophet/docs/quick_start.html#python-api**

Data source:

**https://www.kaggle.com/competitions/demand-forecasting-kernels-only**








### **Setting up Spark environment variables and installing dependencies**

In [13]:
# Importing the warnings module

import warnings
warnings.filterwarnings("ignore")

import os # Importing the OS module

# Logging level set to WARNING for Prophet Package
import logging

logging.getLogger('prophet').setLevel(logging.ERROR)

# Installing OpenJDK environment for running Spark
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Downloading Apache Spark 3.5.0 with Hadoop quietly
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz

# Extract the Spark Package
!tar xf spark-3.5.0-bin-hadoop3.tgz

# Set JAVA_HOME environment path pointing to where OpenJDK is installed
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Set SPARK_HOME environment path pointing to where Spark is installed
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

In [14]:
# Lists files and directories in current working directory

!ls

drive	     spark-3.5.0-bin-hadoop3	  spark-3.5.0-bin-hadoop3.tgz.1
sample_data  spark-3.5.0-bin-hadoop3.tgz


In [15]:
# Installing the findspark package - necessary to run spark from Jupyter Notebook
!pip install findspark --quiet

# Importing the findspark library
import findspark

# Initializing PySpark - locates Spark installation
findspark.init()

### **A higher level and a slightly more straight forward approach for Spark installation**

In [16]:
# !pip install pyspark

### **Importing dependencies**

In [17]:
# Importing SparkSession from pyspark.sql, to create Spark entry points.

from pyspark.sql import SparkSession

In [18]:
# Calling the builder method and getOrCreate to create or
# retrieving a SparkSession
# spark runs locally with available cores


spark =  SparkSession.builder \
         .master("local[*]") \
         .appName("Spark text analytics") \
         .getOrCreate()

In [19]:
# Elements of the Spark session
spark

In [20]:
# Installing prophet for time series forecast
!pip install prophet --quiet

# Importing plotly for advanced interactive visualization
!pip install plotly --quiet

# Importing statsmodels package for statistical computations
!pip install statsmodels --quiet

In [21]:
# Importing pandas and numpy for data cleaning and preprocessing
import pandas as pd
import numpy as np

# Importing matplotlib for data visualization
import matplotlib.pyplot as plt

# Importing datetime for providing DateTime data type
import datetime

# Importing seasonal_decompose for extracting Seasonal, trend components
from statsmodels.tsa.seasonal import seasonal_decompose

# Importing required libraries for data manipulation
import pyspark.sql.functions as F
from pyspark.sql.functions import isnan, count, col, when

# Importing Prophet library for time series forecasting
from prophet import Prophet

# Importing Plotly library for interactive visualization
import plotly.express as px

# Setting pandas backend plotting to plotly
pd.options.plotting.backend ="plotly"

### **Importing the dataset**

In [22]:
# Dependencies to import required files from Google Drive to Google Colab

from google.colab import drive

In [23]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Importing the dataset to Spark DataFrame

sdf = spark.read.csv("/content/drive/MyDrive/train_forecast.csv", header = True, inferSchema = "true")

### **Analysis of the dataset with PySpark**

In [25]:
# Print the schema of the Spark DataFrame in a tree format using .printSchema()
# method

sdf.printSchema()

root
 |-- date: date (nullable = true)
 |-- store: integer (nullable = true)
 |-- item: integer (nullable = true)
 |-- sales: integer (nullable = true)



In [26]:
# Display the first five rows of the Spark DataFrame using df.show() method

sdf.show(n = 5, truncate = False)

+----------+-----+----+-----+
|date      |store|item|sales|
+----------+-----+----+-----+
|2013-01-01|1    |1   |13   |
|2013-01-02|1    |1   |11   |
|2013-01-03|1    |1   |14   |
|2013-01-04|1    |1   |13   |
|2013-01-05|1    |1   |10   |
+----------+-----+----+-----+
only showing top 5 rows



In [27]:
# Display the shape the Spark dataframe using .count() method and .columns
# attribute

print("Shape of the dataframe is", (sdf.count(), len(sdf.columns)))

Shape of the dataframe is (913000, 4)


In [28]:
# Assess the presence of duplicate rows - Data Cleaning

distinctDF = sdf.distinct()

print("Distinct Count:" +str(distinctDF.count())) # display the count of distinct rows
distinctDF.show(n = 10, truncate = False)

# Since Distinct count matches the number of rows in the dataframe, the dataframe
# does not have duplicate values

Distinct Count:913000
+----------+-----+----+-----+
|date      |store|item|sales|
+----------+-----+----+-----+
|2014-02-14|1    |1   |20   |
|2015-07-31|1    |1   |29   |
|2015-08-18|1    |1   |18   |
|2016-01-18|1    |1   |7    |
|2016-03-18|1    |1   |21   |
|2017-08-14|1    |1   |26   |
|2017-09-09|1    |1   |28   |
|2017-11-03|1    |1   |18   |
|2013-06-10|2    |1   |16   |
|2013-08-27|2    |1   |29   |
+----------+-----+----+-----+
only showing top 10 rows



In [29]:
# Assess presence of null values using isNull(), count() and filter() methods
# for data cleaning

for column in sdf.columns:
  nu_count = sdf.filter(col(column).isNull()).count()

  if nu_count > 0:
    print(f"Column {column} has {nu_count} null values:")
    df.filter(col(column).isNull()).show()

  else:
    print(f"Column {column} has no null values")

Column date has no null values
Column store has no null values
Column item has no null values
Column sales has no null values


### **Filtering by date and aggregation function**

In [30]:
# Extract the Earliest and latest date in the dateframe sdf

earliest_date = F.min('date')
latest_date = F.max('date')

df_date = sdf.select(earliest_date, latest_date)
df_date.show()



+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2013-01-01|2017-12-31|
+----------+----------+



In [31]:
# Filter() method with between condition on the date column
# converting spark dataframe sdf to pandas dataframe to display the data

df_1 = sdf.filter("date between '2013-01-01' and  '2013-12-31'")

df_filter = df_1.toPandas()
print("First Five Rows of the dataframe: \n", df_filter.head())
print("Five Rows from the end of the dataframe: \n",df_filter.tail())

First Five Rows of the dataframe: 
          date  store  item  sales
0  2013-01-01      1     1     13
1  2013-01-02      1     1     11
2  2013-01-03      1     1     14
3  2013-01-04      1     1     13
4  2013-01-05      1     1     10
Five Rows from the end of the dataframe: 
               date  store  item  sales
182495  2013-12-27     10    50     53
182496  2013-12-28     10    50     53
182497  2013-12-29     10    50     66
182498  2013-12-30     10    50     29
182499  2013-12-31     10    50     47


In [32]:
# Use of aggregation function to categorize store by count
# number of occurrances of each store

sdf.select(['store']).groupby('store').agg({'store':'count'}).show()

+-----+------------+
|store|count(store)|
+-----+------------+
|    1|       91300|
|    6|       91300|
|    3|       91300|
|    5|       91300|
|    9|       91300|
|    4|       91300|
|    8|       91300|
|    7|       91300|
|   10|       91300|
|    2|       91300|
+-----+------------+



In [33]:

# Create a Temporary view called sales_sdf

sdf.createOrReplaceTempView("sales_sdf")

In [34]:
# SQL Query for categorizing by store count (number of occurrances of each store)
# in the sales_sdf dataframe

spark.sql(""" SELECT store, count(store) FROM sales_sdf
              GROUP BY store ORDER BY store DESC """).show(n=10)

+-----+------------+
|store|count(store)|
+-----+------------+
|   10|       91300|
|    9|       91300|
|    8|       91300|
|    7|       91300|
|    6|       91300|
|    5|       91300|
|    4|       91300|
|    3|       91300|
|    2|       91300|
|    1|       91300|
+-----+------------+



In [35]:
# SQL Query : Aggregating by sum(sales) for each store in descending order of
# date :
# group the data by 'store' and 'date', sum to get the total sale per store
# each date


spark.sql(""" SELECT date, store, sum(sales) FROM sales_sdf GROUP BY
          store, date ORDER BY store DESC, date DESC """).show(n=20)

+----------+-----+----------+
|      date|store|sum(sales)|
+----------+-----+----------+
|2017-12-31|   10|      2939|
|2017-12-30|   10|      2742|
|2017-12-29|   10|      2687|
|2017-12-28|   10|      2429|
|2017-12-27|   10|      2221|
|2017-12-26|   10|      2314|
|2017-12-25|   10|      2007|
|2017-12-24|   10|      2833|
|2017-12-23|   10|      2740|
|2017-12-22|   10|      2597|
|2017-12-21|   10|      2508|
|2017-12-20|   10|      2296|
|2017-12-19|   10|      2321|
|2017-12-18|   10|      1945|
|2017-12-17|   10|      2889|
|2017-12-16|   10|      2730|
|2017-12-15|   10|      2720|
|2017-12-14|   10|      2470|
|2017-12-13|   10|      2342|
|2017-12-12|   10|      2398|
+----------+-----+----------+
only showing top 20 rows



### **Converting the Spark DataFrame to Pandas DataFrame to enable interactive visualization using Plotly Library**

To visualize the dataframe with Plotly, it is essential to convert the PySpark dataframe to a Pandas dataframe - since PySpark doesn't natively support Plotly for visualizations

In [36]:
# Converting Spark dataframe sdf to Pandas dataframe using toPandas() method
df = sdf.toPandas()

In [37]:
# Convert the date column of the pandas dataframe to_datetime() method

df['date'] = pd.to_datetime(df['date'])

In [38]:
# Extract the month from the date column (datetime object) using dt in Pandas
df['month'] = df['date'].dt.month

In [39]:
# Extract the year from the date column (datetime object) using dt in Pandas
df['year'] = df['date'].dt.year


In [40]:
# Using .replace() method to rename entries in the month column
# Mapping each value in integer to abbreviated month in string

df['month'].replace({1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun", 7:"July", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12: "Dec"}, inplace = True)

In [41]:
# Plot of sales (output) column in the dataframe for each date corresponding
# to each store using a scatter plot
# the backend of the plot is set to use the plotly library, which effective maps
# the store to corresponding color - allows for intuitive visual distinction


plt = df.plot.scatter(x = 'date', y = "sales", color = 'store')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [42]:
# Boxplot of sales (output) column in the dataframe, each year using plotly library
# Upon closer examination of the trend, we seen an increase in median sale each year

fig = px.box(df, y = 'sales', x = 'year')
fig.show()

# let us analyze seasonality within the data

Output hidden; open in https://colab.research.google.com to view.

### **Understand the trend and seasonality for an item and a store**



In [43]:
# Filtering using .loc property to extract data corresponding to store 2 and the item 2
# Set the index of the dataframe to the date column using set_index() method
# display the first five rows of the dataframe

df_1 = df.loc[(df['store']== 2) & (df['item'] == 2)]

df_1.set_index('date', inplace = True)
df_1.head()

,store,item,sales,month,year
date,,,,,
2013-01-01,2,2,41,Jan,2013
2013-01-02,2,2,33,Jan,2013
2013-01-03,2,2,46,Jan,2013
2013-01-04,2,2,50,Jan,2013
2013-01-05,2,2,44,Jan,2013


In [44]:
# Analyze the trend using box plot corresponding to store 2 and the item 2 in the dataframe
# using Plotly library
# Allows for visual understanding of the trend which shows an increase in median
# sale each year

fig = px.box(df_1, y = 'sales', x='year')
fig.show()

In [45]:
# Create a Pandas DataFrame called analytics with a copy() of the sales column in df_1 dataframe
# with the data corresponding to store 2 and the item 2
# using the copy() method allows to make changes to the analytics dataframe without
# changing the dataframe df_1

analytics = df_1[["sales"]].copy()

In [46]:
# set the index of analytics dataframe to the datatime object using pd.to_datetime()
# method essential for time series analysis

analytics.index = pd.to_datetime(analytics.index)

In [47]:
# extracting the trend and seasonality component with an additive model
# using seasonal_decompose() function from statsmodels.tsa.seasonal library
# corresponding to the sales column of analytics dataframe

model_1 = seasonal_decompose(analytics, model = "additive")
trend = model_1.trend
seasonality = model_1.seasonal
res = model_1.resid

In [48]:
# Plot of the trend component changes as a function of time - shows an increasing
# overall trend until July (maximum around July) followed by a
# decreasing overall trend for the remaining time of the year

fig = trend.plot()
fig.show()

In [49]:
  # Plot of the seasonality component as a function of time
  # Observing one complete cycle every seven days - shows strong weekly seasonality

  fig = seasonality.plot()
  fig.show()



### **Prophet model is suitable for historical time series data with strong seasonal effects**


- Prophet is an open source model specifically for time series forecasting

- Prophet forecasts data based on an additive model which captures the non-linear trends, fit with seasonality effects

- Works optimally with time series data with strong seasonality effects and is effective with multiple seasons of historical data available for analysis

- Doesn't require a lot of cleaning on the data

- https://facebook.github.io/prophet/

In [50]:
# Preprocessing the data for the Prophet model
# Reset the index of df_1 dataframe which is a requirement for Prophet Model
# Creating model_df dataframe with date, sales column -

df_1.reset_index(inplace = True)
model_df = df_1[['date', 'sales']]

In [51]:
# Display the datatype of sales and date column in df_1 dataframe

model_df.dtypes

date     datetime64[ns]
sales             int32
dtype: object

In [52]:
# Renaming the date column to 'ds'and the output column to 'y' as required by the
# Prophet Model
# Display the first five rows of model_df dataframe

model_df.columns = ['ds', 'y']
model_df.head()

,ds,y
0,2013-01-01,41
1,2013-01-02,33
2,2013-01-03,46
3,2013-01-04,50
4,2013-01-05,44


In [53]:
# Split the model_df dataframe into training and test datasets
# with training dataset to include data until '2017-10-31' and test dataset
# includes data from '2017-11-01'

train = model_df.loc[model_df['ds'] <= '2017-10-31']
test = model_df.loc[model_df['ds'] >= '2017-11-01']

In [54]:
# Initializing the Prophet model for time series training
m  = Prophet()

In [55]:
# Display the end of the train dataset verifies data
# until 31st October 2017
train.tail()

,ds,y
1760,2017-10-27,82
1761,2017-10-28,97
1762,2017-10-29,100
1763,2017-10-30,58
1764,2017-10-31,72


In [56]:
# Model training on the training dataset
m.fit(train)

DEBUG:cmdstanpy:input tempfile: /tmp/tmpnun7w1qb/3it77mox.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpnun7w1qb/hbicbqwl.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.10/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=71440', 'data', 'file=/tmp/tmpnun7w1qb/3it77mox.json', 'init=/tmp/tmpnun7w1qb/hbicbqwl.json', 'output', 'file=/tmp/tmpnun7w1qb/prophet_modelot_b_pnn/prophet_model-20231214113822.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
11:38:22 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
11:38:22 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


In [57]:
# Creating make_future_dataframe for a 61 day forecast

future = m.make_future_dataframe(periods = 61)

In [58]:
# Making Predictions with the trained Prophet Model

predictions = m.predict(future)

In [59]:
# Extracting the date ('ds'), predictions 'yhat' with the confidence interval
# provided by 'yhat_upper' and 'yhat_lower'

forecast = predictions[['ds','yhat','yhat_lower','yhat_upper']]

In [60]:
# Display the end of the forecast dataframe

forecast.tail()

,ds,yhat,yhat_lower,yhat_upper
1821,2017-12-27,60.065342,48.006174,73.552298
1822,2017-12-28,64.637809,51.196994,76.013162
1823,2017-12-29,70.001888,57.680834,82.239665
1824,2017-12-30,73.452020,61.587387,86.453082
1825,2017-12-31,79.374830,66.430687,90.970128


In [61]:
# Saving the predictions to the test dataframe for interactive visualization
# with the confidence intervals

test['y_hat'] = forecast[forecast['ds'] >= '2017-11-01']['yhat']
test['yhat_lower'] = forecast[forecast['ds'] >= '2017-11-01']['yhat_lower']
test['yhat_upper'] = forecast[forecast['ds'] >= '2017-11-01']['yhat_upper']

In [62]:
# Display the first five rows of the dataframe
test.head()

,ds,y,y_hat,yhat_lower,yhat_upper
1765,2017-11-01,84,81.416967,67.951022,93.362418
1766,2017-11-02,93,86.672559,74.085594,99.654209
1767,2017-11-03,78,92.733208,80.217856,104.829952
1768,2017-11-04,100,96.880872,84.743004,109.293339
1769,2017-11-05,103,103.488522,90.468309,115.534380


In [63]:
# Using plotly.graph_objects for interactive visualization
# to compare ground truth with predictions and confidence intervals

import plotly.graph_objs as go
fig = go.Figure([
    go.Scatter(
        name = "Ground Truth",
        x = test["ds"],
        y = test["y"],
        mode = "markers",
        marker = dict(color = "MediumPurple",
                      line = dict(width = 1))
    ),
    go.Scatter(
        name = "Prediction",
        x = test["ds"],
        y = test["y_hat"],
        mode = "lines",
        line = dict(color = "black"),
    ),
    go.Scatter(
        name = "Upper Bound",
        x = test["ds"],
        y = test["yhat_upper"],
        mode = "lines",
        marker = dict(color="#444"),
        line = dict(width=0.5),
        showlegend = False
    ),
    go.Scatter(
        name = "Lower Bound",
        x = test["ds"],
        y = test["yhat_lower"],
        mode = "lines",
        marker = dict(color="#444"),
        line = dict(width=0.5),
        fillcolor = "rgba(68, 68, 68, 0.3)",
        fill = "tonexty",
        showlegend = False
    )
])

fig.update_layout(
    yaxis_title = "Forecast",
    title  = "Forecast for 61 days with confidence interval (Error - Band)",
    hovermode = "x"
)

fig.show()

# https://plotly.com/python/graph-objects/

### **Quiz**

1. Please make predictions for a 30-day forecast using the Prophet model.

2. Please display the dataframe with the 30-day predictions and the confidence interval using the df.tail() method.



In [64]:
# Creating make_future_dataframe for a 30 day forecast

future = m.make_future_dataframe(periods = 30)

In [65]:
# Making Predictions with the trained Prophet Model

predictions = m.predict(future)

In [66]:
# Extracting the date ('ds'), predictions 'yhat' with the confidence interval
# provided by 'yhat_upper' and 'yhat_lower'

forecast = predictions[['ds','yhat','yhat_lower','yhat_upper']]

In [67]:
# Display the end of the forecast dataframe

forecast.tail()

,ds,yhat,yhat_lower,yhat_upper
1790,2017-11-26,96.300103,84.468112,108.745406
1791,2017-11-27,64.666455,51.719618,75.567215
1792,2017-11-28,73.992356,61.757750,86.448233
1793,2017-11-29,72.975767,60.215094,84.213845
1794,2017-11-30,76.463721,64.124660,89.905194
